# The scvd.store corpus, recomputed

Fetches the signed x402 corpus (`/corpus.json` and every snapshot), recomputes each snapshot's digest and the chain, checks the ed25519 signatures against the published key, and counts doors per week **with their denominators**. Run top to bottom. Nothing here trusts the store's own arithmetic: every number below is derived from the bytes you fetch.

What this is not: a ranking. A verdict is what one probe saw from one vantage at one moment; counts here travel with the number of doors probed.

In [ ]:
import hashlib, json, urllib.request

BASE = "https://scvd.store"

def get(path):
    with urllib.request.urlopen(urllib.request.Request(f"{BASE}{path}", headers={"Accept": "application/json"})) as r:
        return json.load(r)

index = get("/corpus.json")
print(index["what_this_is"])
print("entries:", index["entries"])

In [ ]:
# Every snapshot, by its stable address.
records = [get(f"/corpus/{row['sequence']}.json") for row in index["index"]]
len(records)

In [ ]:
# Recompute each digest over the canonical snapshot (fixed field order, no whitespace) and walk the chain.
FIELDS = ["version", "sequence", "taken_at", "previous_digest", "source", "week", "round"]

def canonical(snapshot):
    ordered = {k: snapshot[k] for k in FIELDS}
    return json.dumps(ordered, separators=(",", ":"), ensure_ascii=False)

previous = None
for record in records:
    snap = record["snapshot"]
    digest = hashlib.sha256(canonical(snap).encode("utf-8")).hexdigest()
    assert digest == record["digest"], (snap["sequence"], digest, record["digest"])
    assert snap["previous_digest"] == previous, snap["sequence"]
    previous = digest
print("chain holds:", len(records), "snapshots, every digest recomputed and linked")

In [ ]:
# Signatures, against the published key (not the one in the document). Needs: pip install pynacl
try:
    from nacl.signing import VerifyKey
    key_doc = get("/.well-known/scvd-signing-key")
    public_key = bytes.fromhex(key_doc["public_key"] if "public_key" in key_doc else key_doc["current"]["public_key"])
    for record in records:
        VerifyKey(public_key).verify(canonical(record["snapshot"]).encode("utf-8"), bytes.fromhex(record["signature"]))
    print("every signature verifies against", key_doc.get("key_id", "the published key"))
except ImportError:
    print("pynacl not installed; signatures not checked in this run")

In [ ]:
# Counts with denominators, per week. Never a ratio: the denominator rides beside every count.
for record in records:
    hosts = record["snapshot"]["round"].get("hosts", [])
    probed = [h for h in hosts if h.get("verdict") != "not_probed"]
    ready = [h for h in probed if h.get("verdict") == "ready"]
    print(record["snapshot"]["week"], f"ready {len(ready)} of {len(probed)} probed, {len(hosts)} listed")

Compare any line above with `/corpus/round/{week}` or `/corpus/brief`: they are derived from the same rows, and this notebook owes the store nothing on trust.